# Day 6 — 基本面特征工程

**论文六大维度覆盖**: 估值与成长 / 公司基本面 / 投资 / 盈利

**处理策略**:
- 季度报告数据 forward-fill 到每日 (避免未来信息泄露: 用报告日前一季度的数据)
- 构造比率特征 (归一化使跨股票可比)
- 增长率特征 (同比、环比)

In [ ]:
import pandas as pd
import numpy as np
import os

OUT_DIR = r"C:\Users\1\Desktop\项目\stock-data"
pd.set_option("display.max_columns", 100)

In [ ]:
# ==========================================
# 第1步: 读取季度基本面 + 股价数据
# ==========================================

FUNDA_PATH = os.path.join(OUT_DIR, "day5_quarterly_fundamentals.csv")

if not os.path.exists(FUNDA_PATH):
    raise FileNotFoundError(
        f"未找到 {FUNDA_PATH}\n"
        "请先运行 day5_fundamental_scraper.ipynb 抓取基本面数据, "
        "或确保 day5_quarterly_fundamentals.csv 存在。\n"
        "如果不需要基本面特征, 可以跳过 day5 → day6 → day7, "
        "直接从 day4 → day8 开始。"
    )

df_q = pd.read_csv(FUNDA_PATH)
df_q["report_date"] = pd.to_datetime(df_q["report_date"])

# 读取价格数据以获取日期索引
df_price = pd.read_csv(os.path.join(OUT_DIR, "day4_features.csv"))
df_price["Date"] = pd.to_datetime(df_price["Date"])

print(f"季度数据: {df_q.shape}")
print(f"报告日期范围: {df_q['report_date'].min()} ~ {df_q['report_date'].max()}")
print(f"日度价格数据: {df_price.shape}")
print(f"价格日期范围: {df_price['Date'].min()} ~ {df_price['Date'].max()}")

In [ ]:
# ==========================================
# 第2步: 季度基本面 -> 日度 (前向填充, 防未来信息)
#
# 关键: 假设财报在报告日后30天才可用 (保守估计)
# 每个交易日的特征 = 最近一次 "report_date + 30天" 之前的财报数据
# ==========================================

LAG_DAYS = 30  # 财报发布延迟 (保守)

# 为每个quarterly数据行计算 "可用日期" = report_date + LAG_DAYS
df_q["available_date"] = df_q["report_date"] + pd.Timedelta(days=LAG_DAYS)

# 构建 symbol-date 网格, 填充最近可用的季度数据
dates = df_price[["Date", "symbol"]].drop_duplicates()

# 使用merge_asof: 对每个(symbol, Date), 找到在Date之前的最新available_date
df_q_sorted = df_q.sort_values("available_date")

df_daily_funda = pd.merge_asof(
    df_price[["Date", "symbol"]].sort_values("Date"),
    df_q_sorted,
    left_on="Date",
    right_on="available_date",
    by="symbol",
    direction="backward"  # 只用Date之前的数据
)

print(f"合并后日度基本面: {df_daily_funda.shape}")
print(f"NaN比例:")
for c in df_daily_funda.columns:
    pct = df_daily_funda[c].isna().mean()
    if pct > 0.1:
        print(f"  {c}: {pct:.1%}")

In [ ]:
# ==========================================
# 第3步: 估值与成长特征 (Valuation & Growth)
# ==========================================

print("构造估值与成长特征...")

# 提前计算市值 (用 close * 隐含股数 代理, 或用info里的marketCap fill)
# 此处用每季度的数据构造比率

# 估值比率 (从季度数据计算)
# 市值代理 = close * 股数(从total_assets和股价反推不可行)
# 改用: 企业价值/收入, 资产市值比等

# 成长率 (季度同比 YoY)
for col in ["is_total_revenue", "is_net_income", "is_ebitda", "is_operating_income"]:
    if col in df_daily_funda.columns:
        # 按symbol分组, 对同一季度同比 (4个季度前)
        df_daily_funda[f"{col}_yoy"] = (
            df_daily_funda.groupby("symbol")[col]
            .pct_change(4)  # 4 quarters = 1 year
        )

print("估值与成长特征完成")

In [ ]:
# ==========================================
# 第4步: 公司基本面特征 (Fundamentals)
# ==========================================

print("构造公司基本面特征...")

# ROA = Net Income / Total Assets
if "is_net_income" in df_daily_funda.columns and "bs_total_assets" in df_daily_funda.columns:
    df_daily_funda["roa"] = (
        df_daily_funda["is_net_income"] / df_daily_funda["bs_total_assets"]
    )

# ROE = Net Income / Total Equity
if "is_net_income" in df_daily_funda.columns and "bs_total_equity_gross_minority_interest" in df_daily_funda.columns:
    df_daily_funda["roe"] = (
        df_daily_funda["is_net_income"] / df_daily_funda["bs_total_equity_gross_minority_interest"]
    )

# 负债率 = Total Liabilities / Total Assets
if "bs_total_liabilities_net_minority_interest" in df_daily_funda.columns and "bs_total_assets" in df_daily_funda.columns:
    df_daily_funda["leverage"] = (
        df_daily_funda["bs_total_liabilities_net_minority_interest"] / df_daily_funda["bs_total_assets"]
    )

# 流动比率 = Current Assets / Current Liabilities
# (Current Liabilities = Total Liabilities - Long Term Debt)
if "bs_total_current_assets" in df_daily_funda.columns:
    # 简单代理: Working Capital / Total Assets
    if "bs_working_capital" in df_daily_funda.columns:
        df_daily_funda["working_capital_ratio"] = (
            df_daily_funda["bs_working_capital"] / df_daily_funda["bs_total_assets"]
        )

print("公司基本面特征完成")

In [ ]:
# ==========================================
# 第5步: 投资特征 (Investment)
# ==========================================

print("构造投资特征...")

# 总资产增长率 (YoY)
if "bs_total_assets" in df_daily_funda.columns:
    df_daily_funda["asset_growth_yoy"] = (
        df_daily_funda.groupby("symbol")["bs_total_assets"].pct_change(4)
    )

# CAPEX / Revenue (资本支出占收入比)
if "cf_capital_expenditure" in df_daily_funda.columns and "is_total_revenue" in df_daily_funda.columns:
    # CAPEX是负值(支出), 取绝对值
    df_daily_funda["capex_to_revenue"] = (
        -df_daily_funda["cf_capital_expenditure"] / df_daily_funda["is_total_revenue"]
    )

# CAPEX / Total Assets
if "cf_capital_expenditure" in df_daily_funda.columns and "bs_total_assets" in df_daily_funda.columns:
    df_daily_funda["capex_to_assets"] = (
        -df_daily_funda["cf_capital_expenditure"] / df_daily_funda["bs_total_assets"]
    )

print("投资特征完成")

In [ ]:
# ==========================================
# 第6步: 盈利特征 (Profitability)
# ==========================================

print("构造盈利特征...")

# 毛利率 = Gross Profit / Revenue
if "is_gross_profit" in df_daily_funda.columns and "is_total_revenue" in df_daily_funda.columns:
    df_daily_funda["gross_margin"] = (
        df_daily_funda["is_gross_profit"] / df_daily_funda["is_total_revenue"]
    )

# 营业利润率 = Operating Income / Revenue
if "is_operating_income" in df_daily_funda.columns and "is_total_revenue" in df_daily_funda.columns:
    df_daily_funda["operating_margin"] = (
        df_daily_funda["is_operating_income"] / df_daily_funda["is_total_revenue"]
    )

# 净利润率 = Net Income / Revenue
if "is_net_income" in df_daily_funda.columns and "is_total_revenue" in df_daily_funda.columns:
    df_daily_funda["net_margin"] = (
        df_daily_funda["is_net_income"] / df_daily_funda["is_total_revenue"]
    )

# EBITDA Margin
if "is_ebitda" in df_daily_funda.columns and "is_total_revenue" in df_daily_funda.columns:
    df_daily_funda["ebitda_margin"] = (
        df_daily_funda["is_ebitda"] / df_daily_funda["is_total_revenue"]
    )

# EPS增长 (YoY)
if "is_diluted_eps" in df_daily_funda.columns:
    df_daily_funda["eps_growth_yoy"] = (
        df_daily_funda.groupby("symbol")["is_diluted_eps"].pct_change(4)
    )

print("盈利特征完成")

In [ ]:
# ==========================================
# 第7步: 构建最终基本面特征列
# ==========================================

# 挑选基本面特征 (排除原始金额字段, 只保留比率和增长率)
funda_features = [
    c for c in df_daily_funda.columns
    if c not in ["Date", "symbol", "report_date", "available_date"]
    and not c.startswith("bs_")
    and not c.startswith("is_")
    and not c.startswith("cf_")
]

print(f"基本面特征 ({len(funda_features)} 个):")
for f in funda_features:
    print(f"  {f}")

# 只保留 [Date, symbol] + 基本面上特征
df_funda_out = df_daily_funda[["Date", "symbol"] + funda_features].copy()
print(f"\n输出维度: {df_funda_out.shape}")

In [ ]:
# ==========================================
# 第8步: 保存
# ==========================================

df_funda_out.to_csv(
    os.path.join(OUT_DIR, "day6_fundamental_features.csv"),
    index=False, encoding="utf-8-sig"
)

print("Day6 完成! 已保存 day6_fundamental_features.csv")